# V36 — distil the Phase-1 dictionary teacher into one always-on Demucs

This notebook uses `submission_Track2_V23_RECOVER_BALANCED_a8dfcafd.zip`
**only as an offline teacher**. It pairs changed Phase-1 outputs with their
original inputs, rejects the unchanged natural clips, dynamically remixes the
recovered residuals, and fine-tunes the V35 epoch-3 neural enhancer.

The final Phase-2 inference path contains no dictionary, correlation search,
natural/synthetic detector, filename rule, or bypass. Every clip is processed
by the same neural model.

The notebook is restart-safe: teacher extraction, checkpoints, ASR decodes and
submission artifacts are cached separately from V34/V35.


## 0 — Preflight before teacher extraction or training


## 1 — Configuration and exact audio/submission contract


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
from pathlib import Path
import os, io, re, gc, json, math, wave, shutil, random, hashlib
import zipfile, unicodedata, time

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import torch
from tqdm.auto import tqdm
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download, snapshot_download
from IPython.display import display, Audio


ROOT = Path("/content/drive/MyDrive/PHD/Datathon")
VAL_ROOT = ROOT / "validation"
META_PATH = VAL_ROOT / "validationMetadata.json"
SOURCE_AUDIO = ROOT / "input_data.zip"
TRANSCRIPTS_JSONL = ROOT / "transcripts.jsonl"

V34_RUN = ROOT / "track2_phase2_v34_noise_adaptive_residual_demucs" / "run_v1"
V35_RUN = ROOT / "track2_phase2_v35_asr_constrained_adaptive_demucs" / "run_v1"

OUT = ROOT / "track2_phase2_v36_phase1_teacher_demucs"
LOCAL = Path("/content/track2_phase2_v36_phase1_teacher_demucs")
RUN = OUT / "run_v1"
WORK = LOCAL / "run_v1"

for p in (OUT, LOCAL, RUN, WORK):
    p.mkdir(parents=True, exist_ok=True)


# Exact Phase-1 dictionary-teacher submission supplied by the user.
TEACHER_ZIP = Path(
    "/content/drive/MyDrive/Datathon/"
    "track2_v23_high_recall_missed_event_recovery/"
    "run_a8dfcafd81bd5717/"
    "submission_Track2_V23_RECOVER_BALANCED_a8dfcafd.zip"
)

if not TEACHER_ZIP.is_file():
    raise FileNotFoundError(
        f"Phase-1 teacher submission was not found:\n{TEACHER_ZIP}"
    )


# Prefer the ASR-constrained V35 model; use V34 only as a fallback.
SOURCE_CANDIDATES = [
    V35_RUN / "asrft_epoch_03.pt",
    V34_RUN / "adaptive_epoch_02.pt",
]

SOURCE_CHECKPOINT = next(
    (p for p in SOURCE_CANDIDATES if p.is_file()),
    None,
)

if SOURCE_CHECKPOINT is None:
    raise FileNotFoundError(
        "Neither source checkpoint was found:\n"
        + "\n".join(str(p) for p in SOURCE_CANDIDATES)
    )


# Set this only if automatic Phase-1 input ZIP matching fails.
PHASE1_INPUT_OVERRIDE = None


SR = 16000
SEED = 517

NATURAL_DEV_LIMIT = 100
NATURAL_MONITOR_LIMIT = 100

SEGMENT = 51200
BATCH_SIZE = 4
FINETUNE_EPOCHS = 4
DRAWS_PER_EPOCH = 4000

LR_ADAPTER = 2e-5
LR_DECODER = 1e-6
LR_FULL = 2e-7

DIRECT_PAIR_PROB = 0.45
IDENTITY_PROBABILITY = 0.05

NOISE_SISDR_WEIGHT = 0.30
COMPLEX_NOISE_WEIGHT = 0.25
NOISE_WAVE_WEIGHT = 0.10
OUTSIDE_IDENTITY_WEIGHT = 0.20

CHUNK_SAMPLES = 51200
CHUNK_HOP = 25600
EXPECTED_PHASE2_CLIPS = 3178

SIGN_ENSEMBLE = True
GLOBAL_ALPHA = 1.0
RUN_PHASE2 = True

CONFIRM_RULES_ALLOW_PHASE1_TEACHER_DISTILLATION = True
CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING = True


ASR_REPO = "ARTPARK-IISc/SraVaani-1.0"

ASR_PIN_CANDIDATES = [
    Path(
        "/content/drive/MyDrive/Datathon/"
        "track2_v6_direct_reconstruction/"
        "sravaani_revision_pin.json"
    ),
    ROOT / "track2_v6_direct_reconstruction" / "sravaani_revision_pin.json",
]

ASR_PIN = next(
    (p for p in ASR_PIN_CANDIDATES if p.is_file()),
    ASR_PIN_CANDIDATES[-1],
)

DEVICE = torch.device("cuda")
IMPLEMENTATION = "v36_phase1_dictionary_teacher_demucs_v1"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

if not CONFIRM_RULES_ALLOW_PHASE1_TEACHER_DISTILLATION:
    raise RuntimeError(
        "Confirm that Phase-1 submission outputs may be used "
        "as offline training targets."
    )

required_paths = [
    TEACHER_ZIP,
    SOURCE_CHECKPOINT,
    META_PATH,
    SOURCE_AUDIO,
    TRANSCRIPTS_JSONL,
    ASR_PIN,
]

missing_paths = [str(p) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Required files are missing:\n" + "\n".join(missing_paths)
    )

print("V36 run:             ", RUN)
print("Teacher ZIP:         ", TEACHER_ZIP)
print("Source checkpoint:   ", SOURCE_CHECKPOINT)
print("Phase-1 input mode:  ", "automatic matching")
print("SraVaani revision:   ", ASR_PIN)
print("Configuration check: PASS")

V36 run:              /content/drive/MyDrive/PHD/Datathon/track2_phase2_v36_phase1_teacher_demucs/run_v1
Teacher ZIP:          /content/drive/MyDrive/Datathon/track2_v23_high_recall_missed_event_recovery/run_a8dfcafd81bd5717/submission_Track2_V23_RECOVER_BALANCED_a8dfcafd.zip
Source checkpoint:    /content/drive/MyDrive/PHD/Datathon/track2_phase2_v35_asr_constrained_adaptive_demucs/run_v1/asrft_epoch_03.pt
Phase-1 input mode:   automatic matching
SraVaani revision:    /content/drive/MyDrive/Datathon/track2_v6_direct_reconstruction/sravaani_revision_pin.json
Configuration check: PASS


In [3]:
def jsonable(x):
    if isinstance(x, dict): return {str(k):jsonable(v) for k,v in x.items()}
    if isinstance(x, (list, tuple)): return [jsonable(v) for v in x]
    if isinstance(x, Path): return str(x)
    if isinstance(x, np.ndarray): return jsonable(x.tolist())
    if isinstance(x, np.generic): return jsonable(x.item())
    if isinstance(x, float) and not math.isfinite(x): return None
    return x

def put_json(path, obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    t=p.with_name(p.name+'.tmp')
    with open(t,'w',encoding='utf-8') as f:
        json.dump(jsonable(obj),f,indent=2,ensure_ascii=False,allow_nan=False)
    os.replace(t,p)

def sha_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(4<<20),b''): h.update(b)
    return h.hexdigest()

def sha_json(obj):
    return hashlib.sha256(json.dumps(jsonable(obj),sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

def canonical_pcm(y):
    a=np.asarray(y,np.float64)
    if a.ndim!=1 or not len(a) or not np.isfinite(a).all():
        raise ValueError('Wave must be finite, nonempty, and mono.')
    # Deterministic rounding, not peak normalization. Used for metrics AND export.
    return np.clip(np.rint(a*32768.0),-32768,32767).astype('<i2')

def pcm_roundtrip(y): return canonical_pcm(y).astype(np.float64)/32768.0

def wav_bytes(y, sr=16000):
    buf=io.BytesIO()
    with wave.open(buf,'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
        w.writeframes(canonical_pcm(y).tobytes())
    return buf.getvalue()

def write_pcm(path,y,sr=16000):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    data=wav_bytes(y,sr); tmp=p.with_name(p.name+'.tmp')
    with open(tmp,'wb') as f: f.write(data)
    os.replace(tmp,p)
    return hashlib.sha256(data).hexdigest()

def read_wave_strict(path_or_bytes,sr=16000):
    source=io.BytesIO(path_or_bytes) if isinstance(path_or_bytes,(bytes,bytearray)) else path_or_bytes
    y,fs=sf.read(source,dtype='float32',always_2d=True)
    if fs!=sr or y.shape[1]!=1:
        raise ValueError(f'Expected native {sr}-Hz mono audio, got {fs} Hz / {y.shape[1]} channels. No unvalidated resampling.')
    a=y[:,0]
    if not len(a) or not np.isfinite(a).all(): raise ValueError('Empty or invalid audio.')
    return a

def wav_audit(path_or_bytes,expected_frames,sr=16000):
    data=path_or_bytes if isinstance(path_or_bytes,bytes) else Path(path_or_bytes).read_bytes()
    with wave.open(io.BytesIO(data),'rb') as w:
        got=(w.getframerate(),w.getnchannels(),w.getsampwidth(),w.getnframes(),w.getcomptype())
        wanted=(sr,1,2,int(expected_frames),'NONE')
        if got!=wanted: raise ValueError(f'WAV compliance error: {got} != {wanted}')
        pcm=w.readframes(expected_frames)
        if len(pcm)!=expected_frames*2: raise ValueError('Truncated PCM payload.')
    return hashlib.sha256(data).hexdigest()

class StrictWaveSource:
    """One explicit directory/ZIP, exact IDs; no duplicate stems or inferred paths."""
    def __init__(self,path,required_ids):
        self.path=str(path); self.archive=None; self.index={}
        required_ids=set(map(str,required_ids))
        p=Path(path)
        if p.is_file() and zipfile.is_zipfile(p):
            self.archive=zipfile.ZipFile(p)
            names=[z.filename for z in self.archive.infolist() if not z.is_dir() and z.filename.lower().endswith('.wav') and '__MACOSX' not in z.filename]
        elif p.is_dir(): names=[str(q) for q in p.rglob('*') if q.is_file() and q.suffix.lower()=='.wav']
        else: raise FileNotFoundError(p)
        for name in names:
            cid=Path(name).stem
            if cid not in required_ids: continue
            if cid in self.index: raise ValueError(f'Duplicate waveform ID: {cid} in {p}')
            self.index[cid]=name
        missing=required_ids-set(self.index)
        if missing: raise ValueError(f'{p}: {len(missing)} missing IDs; examples {sorted(missing)[:3]}')
    def bytes(self,cid):
        loc=self.index[cid]
        return self.archive.read(loc) if self.archive is not None else Path(loc).read_bytes()
    def load(self,cid): return read_wave_strict(self.bytes(cid))
    def close(self):
        if self.archive is not None: self.archive.close()

def si_sdr(estimate,reference):
    """Zero-mean SI-SDR, no alignment/resampling; 120-dB numerical ceiling only."""
    x=np.asarray(estimate,np.float64); s=np.asarray(reference,np.float64)
    if x.shape!=s.shape or x.ndim!=1: raise ValueError('SI-SDR length/shape mismatch.')
    if not np.isfinite(x).all() or not np.isfinite(s).all(): raise ValueError('Nonfinite SI-SDR input.')
    x=x-x.mean(); s=s-s.mean(); ss=float(s@s)
    if ss<=1e-18: raise ValueError('Silent reference: SI-SDR undefined, not silently excluded.')
    target=(float(x@s)/ss)*s; e=x-target
    num=float(target@target); den=float(e@e)
    value=10*math.log10(max(num,ss*1e-16)/max(den,ss*1e-12))
    return float(min(120.,value))

def hyp_text(h):
    if isinstance(h,str): return h.strip()
    if isinstance(h,dict) and isinstance(h.get('text'),str): return h['text'].strip()
    if hasattr(h,'text') and isinstance(h.text,str): return h.text.strip()
    if isinstance(h,(list,tuple)) and len(h)==1: return hyp_text(h[0])
    raise TypeError(f'Unrecognized ASR hypothesis type: {type(h)}. Refusing an invented empty transcript.')

def pack_submission(paths,asr,expected_frames,output_zip,sr=16000):
    """Only final audio and matching ASR enter the ZIP; diagnostics/clean refs never do."""
    ids=sorted(paths)
    if set(ids)!=set(asr) or set(ids)!=set(expected_frames): raise ValueError('Submission ID mismatch.')
    for cid in ids:
        if not cid or Path(cid).name!=cid or any(c in cid for c in ('/','\\','\n','\r')): raise ValueError('Unsafe clip ID.')
        h=wav_audit(paths[cid],expected_frames[cid],sr)
        if h!=asr[cid]['wav_sha256']: raise ValueError(f'ASR/audio hash mismatch: {cid}')
        if not isinstance(asr[cid]['text'],str): raise ValueError('ASR text must be a string.')
    p=Path(output_zip); p.parent.mkdir(parents=True,exist_ok=True); tmp=p.with_name(p.name+'.tmp')
    tr=''.join(json.dumps(dict(clip_id=c,text=asr[c]['text']),ensure_ascii=False,allow_nan=False)+'\n' for c in ids).encode('utf-8')
    with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED,compresslevel=9,allowZip64=True) as z:
        z.writestr('transcripts.jsonl',tr)
        for cid in ids: z.write(paths[cid],cid+'.wav')
    with zipfile.ZipFile(tmp) as z:
        wanted={'transcripts.jsonl'}|{c+'.wav' for c in ids}
        if len(z.namelist())!=len(wanted) or set(z.namelist())!=wanted: raise ValueError('Archive contents mismatch.')
        if z.testzip() is not None: raise ValueError('Archive CRC failure.')
        for cid in ids:
            data=z.read(cid+'.wav')
            if wav_audit(data,expected_frames[cid],sr)!=asr[cid]['wav_sha256']: raise ValueError('Packaged audio mismatch.')
        rows=[json.loads(x) for x in z.read('transcripts.jsonl').decode().splitlines()]
        if len(rows)!=len(ids) or {r['clip_id'] for r in rows}!=set(ids): raise ValueError('Packaged transcript mismatch.')
    os.replace(tmp,p)
    return dict(zip_path=str(p),zip_sha256=sha_file(p),clips=len(ids),transcripts_sha256=hashlib.sha256(tr).hexdigest())


## 2 — Reconstruct the fixed real validation splits


In [4]:
def wav_index(root):
    if not root.is_dir(): raise FileNotFoundError(root)
    result={}
    for p in sorted(root.rglob('*')):
        if p.is_file() and p.suffix.lower()=='.wav':
            if p.name in result: raise ValueError(f'Duplicate filename in {root}: {p.name}')
            result[p.name]=p
    return result

def filename_group(name):
    # Dataset filenames end the apparent participant/session token with digits,
    # followed by a numeric utterance token. Retain prefix before that token.
    m=re.match(r'^(.*?\d{6,})_\d+_',Path(name).stem)
    return m.group(1) if m else Path(name).stem

def make_validation_tables(meta_path,root):
    records=json.loads(meta_path.read_text(encoding='utf-8'))
    if not isinstance(records,list): raise ValueError('Expected a JSON list')
    names=[r['segmentFileName'] for r in records]
    if len(names)!=len(set(names)): raise ValueError('Duplicate metadata filenames')
    noisy=wav_index(root/'syntheticNoiseAudio')
    clean=wav_index(root/'syntheticCleanRefAudio')
    natural=wav_index(root/'naturalNoisyAudio')
    pairs=[]; nat=[]
    for r in tqdm(records,desc='Audit validation audio'):
        name=r['segmentFileName']; cid=Path(name).stem
        if Path(name).name!=name: raise ValueError('Expected basename in segmentFileName')
        if not isinstance(r['syntheticData'],bool): raise ValueError('syntheticData must be Boolean')
        common=dict(clip_id=cid,transcript=r.get('transcript',''),language=r.get('language',''),
                    events=r.get('NoiseSubCategoryTimeStamp',[]))
        if r['syntheticData']:
            if name not in noisy or name not in clean: raise FileNotFoundError(f'Missing pair: {name}')
            x=read_wave_strict(noisy[name]); s=read_wave_strict(clean[name])
            if x.shape!=s.shape: raise ValueError(f'Pair length mismatch: {name}')
            if np.array_equal(x,s): raise ValueError(f'Identical noisy/clean pair: {name}')
            if np.mean(s.astype(np.float64)**2)<1e-10: raise ValueError(f'Silent clean target: {name}')
            pairs.append(dict(**common,noisy_path=str(noisy[name]),clean_path=str(clean[name]),
                group_proxy=filename_group(name),frames=len(x),
                noisy_hash=hashlib.sha256(x.tobytes()).hexdigest(),
                clean_hash=hashlib.sha256(s.tobytes()).hexdigest()))
        else:
            if name not in natural: raise FileNotFoundError(f'Missing natural WAV: {name}')
            x=read_wave_strict(natural[name])
            nat.append(dict(**common,noisy_path=str(natural[name]),frames=len(x),
                            noisy_hash=hashlib.sha256(x.tobytes()).hexdigest()))
    # Union grouping proxies and exact clean/noisy duplicates, including cross roles.
    parent=list(range(len(pairs)))
    def find(i):
        while parent[i]!=i:
            parent[i]=parent[parent[i]]; i=parent[i]
        return i
    def union(a,b): parent[find(a)]=find(b)
    seen={}
    for i,r in enumerate(pairs):
        for key in [('proxy',r['group_proxy']),('audio',r['clean_hash']),('audio',r['noisy_hash'])]:
            if key in seen: union(i,seen[key])
            else: seen[key]=i
    groups={}
    for i in range(len(pairs)): groups.setdefault(find(i),[]).append(i)
    if len(groups)<10: raise ValueError('Too few independent groups for a useful split')
    ordered=sorted(groups.values(),key=lambda ids: hashlib.sha256(
        (str(SEED)+'|'+min(pairs[i]['clip_id'] for i in ids)).encode()).hexdigest())
    ng=len(ordered); n_hold=max(1,round(.15*ng)); n_dev=max(1,round(.15*ng))
    for j,ids in enumerate(ordered):
        split='holdout' if j<n_hold else 'dev' if j<n_hold+n_dev else 'train'
        group=min(pairs[i]['clip_id'] for i in ids)
        for i in ids: pairs[i].update(split=split,split_group=group)
    return pd.DataFrame(pairs),pd.DataFrame(nat)

PAIRS,NATURAL=make_validation_tables(META_PATH,VAL_ROOT)
if len(PAIRS)!=695 or len(NATURAL)!=1806:
    raise ValueError(f'Metadata changed: {len(PAIRS)} paired / {len(NATURAL)} natural. Inspect before proceeding.')
DATA_SHA=sha_json(dict(metadata=sha_file(META_PATH),audio=PAIRS[
    ['clip_id','noisy_hash','clean_hash','split']].to_dict('records')))

def natural_sample(df,exclude_ids=(),limit=None):
    rows=df[~df.clip_id.isin(exclude_ids)].copy()
    rows['_key']=rows.clip_id.map(lambda s:hashlib.sha256((str(SEED)+s).encode()).hexdigest())
    rows=rows.sort_values('_key')
    return rows if limit is None else rows.head(limit)
# Prior V27 natural sample is explicitly the development set, not a new holdout.

NAT_DEV=natural_sample(NATURAL,limit=NATURAL_DEV_LIMIT)
NAT_MON=natural_sample(NATURAL,exclude_ids=set(NAT_DEV.clip_id),limit=NATURAL_MONITOR_LIMIT)
# V34 uses the original holdout as the untouched genuine selection set and
# adds the former development split to training. No holdout pair is fitted.
SYN_DEV=PAIRS[PAIRS.split=="holdout"].copy()
SYN_MON=PAIRS[PAIRS.split=="dev"].copy()      # fit diagnostic only; never selects a checkpoint
TRAIN_ROWS=PAIRS[PAIRS.split!="holdout"].copy()
assert set(TRAIN_ROWS.clip_id).isdisjoint(set(SYN_DEV.clip_id))
assert set(NAT_DEV.clip_id).isdisjoint(NAT_MON.clip_id)
print("Genuine training / untouched synthetic selection:",len(TRAIN_ROWS),len(SYN_DEV))
print("Natural development / monitoring:",len(NAT_DEV),len(NAT_MON))


Audit validation audio:   0%|          | 0/2501 [00:00<?, ?it/s]

Genuine training / untouched synthetic selection: 591 104
Natural development / monitoring: 100 100


In [6]:
PHASE1_INPUT_OVERRIDE = Path(
    "/content/drive/MyDrive/Datathon/"
    "track2_v23_high_recall_missed_event_recovery/"
    "run_a8dfcafd81bd5717/"
    "submission_Track2_V23_RECOVER_BALANCED_a8dfcafd.zip"
)

## 3 — Recover changed Phase-1 teacher pairs and exclude untouched natural audio


In [9]:
def clean_text(text):
    text = re.sub(r"<[^>]*>|\[[^\]]*\]", " ", str(text or ""))
    return re.sub(
        r"\s+", " ", unicodedata.normalize("NFC", text)
    ).strip()


# Phase-2 transcripts remain required later for final submission generation.
TRANSCRIPTS = [
    json.loads(line)
    for line in TRANSCRIPTS_JSONL.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

TRANSCRIPT_MAP = {
    str(r["clip_id"]): clean_text(r["text"])
    for r in TRANSCRIPTS
}

if len(TRANSCRIPT_MAP) != EXPECTED_PHASE2_CLIPS:
    raise ValueError(
        f"Expected {EXPECTED_PHASE2_CLIPS} Phase-2 transcripts, "
        f"found {len(TRANSCRIPT_MAP)}"
    )


def zip_wav_index(path):
    with zipfile.ZipFile(path) as z:
        names = [
            name
            for name in z.namelist()
            if name.lower().endswith(".wav")
            and "__MACOSX" not in name
        ]

    result = {}

    for name in names:
        stem = Path(name).stem

        if stem in result:
            raise ValueError(
                f"Duplicate WAV stem {stem} in {path}"
            )

        result[stem] = name

    return result


def flat_wav_index(directory):
    """Index only WAV files directly inside directory—no recursion."""
    directory = Path(directory)

    if not directory.is_dir():
        raise FileNotFoundError(directory)

    result = {}

    for path in sorted(directory.iterdir()):
        if not path.is_file():
            continue

        if path.suffix.lower() != ".wav":
            continue

        stem = path.stem

        if stem in result:
            raise ValueError(
                f"Duplicate WAV stem {stem} in {directory}"
            )

        result[stem] = path

    return result


# Enhanced outputs from the V23 dictionary-based submission.
teacher_index = zip_wav_index(TEACHER_ZIP)

if len(teacher_index) < 1000:
    raise RuntimeError(
        f"Teacher ZIP contains only {len(teacher_index)} WAV files"
    )


# Original Phase-1 noisy WAVs are directly in this directory.
# No recursive Drive search is performed.
PHASE1_INPUT_DIR = Path(
    "/content/drive/MyDrive/PHD/Datathon"
)

input_index = flat_wav_index(PHASE1_INPUT_DIR)

if not input_index:
    raise FileNotFoundError(
        f"No top-level WAV files found in {PHASE1_INPUT_DIR}"
    )


common_ids = sorted(
    set(teacher_index).intersection(input_index)
)

coverage = (
    len(common_ids) / len(teacher_index)
    if teacher_index else 0.0
)

print("Teacher ZIP:             ", TEACHER_ZIP)
print("Phase-1 noisy directory: ", PHASE1_INPUT_DIR)
print("Teacher WAVs:            ", len(teacher_index))
print("Top-level noisy WAVs:    ", len(input_index))
print("Common clip IDs:         ", len(common_ids))
print("Teacher coverage:        ", f"{100 * coverage:.2f}%")


if len(common_ids) < 500:
    teacher_examples = sorted(teacher_index)[:5]
    input_examples = sorted(input_index)[:5]

    print("\nExample teacher IDs:")
    print(*teacher_examples, sep="\n")

    print("\nExample input IDs:")
    print(*input_examples, sep="\n")

    raise RuntimeError(
        f"Only {len(common_ids)} common IDs were found. "
        "Check whether the original Phase-1 WAVs are directly "
        "inside PHASE1_INPUT_DIR."
    )


TEACHER_CACHE = RUN / "phase1_teacher_cache"
MIX_DIR = TEACHER_CACHE / "mixture"
CLEAN_DIR = TEACHER_CACHE / "teacher_clean"

for directory in (TEACHER_CACHE, MIX_DIR, CLEAN_DIR):
    directory.mkdir(parents=True, exist_ok=True)

manifest_path = RUN / "phase1_teacher_pairs.jsonl"


def bytes_wave(raw, label):
    y, sr = sf.read(
        io.BytesIO(raw),
        dtype="float32",
        always_2d=True,
    )

    if sr != SR:
        raise ValueError(
            f"{label}: expected {SR} Hz, found {sr}"
        )

    if y.shape[1] != 1:
        raise ValueError(
            f"{label}: expected mono, "
            f"found {y.shape[1]} channels"
        )

    y = y[:, 0].astype(np.float32)

    if not len(y) or not np.isfinite(y).all():
        raise ValueError(f"{label}: invalid waveform")

    return y


# Load only compatible cached records. This prevents records created
# using an incorrectly selected ZIP from being reused.
existing = {}

if manifest_path.exists():
    for line in manifest_path.read_text(
        encoding="utf-8"
    ).splitlines():
        if not line.strip():
            continue

        record = json.loads(line)

        if (
            record.get("phase1_input")
            == str(PHASE1_INPUT_DIR)
        ):
            existing[record["pair_id"]] = record


rows = []

counts = {
    "common": len(common_ids),
    "unchanged": 0,
    "gain_only": 0,
    "length_mismatch": 0,
    "rejected_gain": 0,
    "rejected_correlation": 0,
    "silent_target": 0,
    "cached": 0,
    "accepted": 0,
}


with zipfile.ZipFile(TEACHER_ZIP) as teacher_zip:
    for k, clip_id in enumerate(
        tqdm(
            common_ids,
            desc="Recover Phase-1 teacher pairs",
        )
    ):
        pair_id = "p1_" + clip_id
        cached = existing.get(pair_id)

        if (
            cached
            and Path(cached["noisy_path"]).is_file()
            and Path(cached["clean_path"]).is_file()
        ):
            rows.append(cached)
            counts["cached"] += 1
            continue

        # x: actual original Phase-1 noisy input.
        x = read_wave_strict(
            input_index[clip_id]
        )

        # y: corresponding enhanced V23 submission output.
        y = bytes_wave(
            teacher_zip.read(teacher_index[clip_id]),
            clip_id + ":teacher",
        )

        if len(x) != len(y):
            counts["length_mismatch"] += 1
            continue

        # Natural clips copied without processing are not targets.
        if np.array_equal(
            canonical_pcm(x),
            canonical_pcm(y),
        ):
            counts["unchanged"] += 1
            continue

        x64 = x.astype(np.float64)
        y64 = y.astype(np.float64)

        # Restore teacher output to the input gain domain.
        gain = float(
            np.dot(x64, y64)
            / (np.dot(y64, y64) + 1e-12)
        )

        if not 0.40 <= gain <= 2.50:
            counts["rejected_gain"] += 1
            continue

        target = (gain * y64).astype(np.float32)
        residual = x - target

        change_rms = float(
            np.sqrt(
                np.mean(
                    residual.astype(np.float64) ** 2
                )
            )
        )

        target_rms = float(
            np.sqrt(
                np.mean(
                    target.astype(np.float64) ** 2
                )
            )
        )

        # Gain-only exports become almost identical here.
        if change_rms < 2e-5:
            counts["gain_only"] += 1
            continue

        if target_rms < 1e-4:
            counts["silent_target"] += 1
            continue

        residual64 = residual.astype(np.float64)
        target64 = target.astype(np.float64)

        correlation = float(
            abs(np.dot(residual64, target64))
            / (
                np.linalg.norm(residual64)
                * np.linalg.norm(target64)
                + 1e-12
            )
        )

        if correlation > 0.75:
            counts["rejected_correlation"] += 1
            continue

        noisy_path = MIX_DIR / f"{pair_id}.wav"
        clean_path = CLEAN_DIR / f"{pair_id}.wav"

        write_pcm(noisy_path, x)
        write_pcm(clean_path, target)

        record = {
            "pair_id": pair_id,
            "source_clip": clip_id,
            "noisy_path": str(noisy_path),
            "clean_path": str(clean_path),
            "frames": len(x),
            "teacher_gain": gain,
            "change_rms": change_rms,
            "residual_speech_correlation": correlation,
            "teacher_zip": str(TEACHER_ZIP),
            "phase1_input": str(PHASE1_INPUT_DIR),
        }

        rows.append(record)
        counts["accepted"] += 1

        if (k + 1) % 100 == 0:
            temporary_manifest = (
                manifest_path.with_suffix(".tmp")
            )

            temporary_manifest.write_text(
                "".join(
                    json.dumps(
                        row,
                        ensure_ascii=False,
                    )
                    + "\n"
                    for row in rows
                ),
                encoding="utf-8",
            )

            os.replace(
                temporary_manifest,
                manifest_path,
            )


temporary_manifest = manifest_path.with_suffix(".tmp")

temporary_manifest.write_text(
    "".join(
        json.dumps(row, ensure_ascii=False) + "\n"
        for row in rows
    ),
    encoding="utf-8",
)

os.replace(
    temporary_manifest,
    manifest_path,
)


TEACHER = pd.DataFrame(rows)

print("\nPhase-1 recovery audit:")
display(
    pd.DataFrame(
        [
            {
                "metric": key,
                "clips": value,
            }
            for key, value in counts.items()
        ]
    )
)


if len(TEACHER) < 500:
    raise RuntimeError(
        f"Only {len(TEACHER)} usable changed teacher pairs "
        f"were recovered. Audit counts: {counts}"
    )

if TEACHER["pair_id"].duplicated().any():
    raise ValueError(
        "Duplicate teacher pair IDs"
    )

print(
    "\nPHASE-1 TEACHER RECOVERY PASS | usable:",
    len(TEACHER),
)

display(
    TEACHER[
        [
            "frames",
            "teacher_gain",
            "change_rms",
            "residual_speech_correlation",
        ]
    ].describe()
)

Teacher ZIP:              /content/drive/MyDrive/Datathon/track2_v23_high_recall_missed_event_recovery/run_a8dfcafd81bd5717/submission_Track2_V23_RECOVER_BALANCED_a8dfcafd.zip
Phase-1 noisy directory:  /content/drive/MyDrive/PHD/Datathon
Teacher WAVs:             5517
Top-level noisy WAVs:     5517
Common clip IDs:          5517
Teacher coverage:         100.00%


Recover Phase-1 teacher pairs:   0%|          | 0/5517 [00:00<?, ?it/s]


Phase-1 recovery audit:


,metric,clips
0,common,5517
1,unchanged,3362
2,gain_only,0
3,length_mismatch,0
4,rejected_gain,0
5,rejected_correlation,0
6,silent_target,0
7,cached,0
8,accepted,2155



PHASE-1 TEACHER RECOVERY PASS | usable: 2155


,frames,teacher_gain,change_rms,residual_speech_correlation
count,2155.000000,2155.000000,2155.000000,2.155000e+03
mean,110697.126682,1.000480,0.056773,2.636016e-09
std,46971.012303,0.012295,0.029196,2.957278e-08
min,22512.000000,0.865918,0.000479,1.007668e-13
25%,76752.000000,0.997153,0.035895,9.568958e-11
50%,98384.000000,1.000013,0.055131,2.154301e-10
75%,136930.000000,1.003178,0.074610,4.538196e-10
max,338860.000000,1.101337,0.164664,1.172475e-06


In [11]:
!pip install denoiser

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.8/49.8 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.0 MB/s eta 0:00:00
  Created wheel for denoiser: filename=denoiser-0.1.5-py3-none-any.whl size=43900 sha256=de8ab921720c26bb9eddbf55f6f624e4613a113d54d3b9b0d4c0a0848175a7a5
  Stored in directory: /root/.cache/pip/wheels/cd/1e/84/e0017338014115353dd95a07e3f7467c8291d0f7cc37d17aef
Successfully built denoiser
  Attempting uninstall: omegaconf
    Found existing installation: omegaconf 2.3.1
    Uninstalling omegaconf-2.3.1:
      Successfully uninstalled omegaconf-2.3.1


## 4 — Restore the V35 noise-adaptive Demucs source checkpoint


In [12]:
from denoiser import pretrained as demucs_pretrained
from denoiser.resample import upsample2,downsample2

base=demucs_pretrained.dns64(pretrained=False)
if base.sample_rate!=SR or base.chin!=1:raise RuntimeError("Unexpected Demucs configuration")
BASELINE_SHA=sha_file(SOURCE_CHECKPOINT)

class SpectralNoiseConditioner(nn.Module):
    def __init__(self,n_fft=512,hop=160,embedding=128):
        super().__init__();self.n_fft=n_fft;self.hop=hop
        self.register_buffer("window",torch.hann_window(n_fft),persistent=False)
        features=2*(n_fft//2+1)
        self.net=nn.Sequential(nn.LayerNorm(features),nn.Linear(features,256),nn.SiLU(),
                               nn.Linear(256,embedding),nn.SiLU())
    def forward(self,wave):
        with torch.amp.autocast(device_type=wave.device.type,enabled=False):
            spec=torch.stft(wave.float(),self.n_fft,hop_length=self.hop,window=self.window.float(),
                            return_complex=True,pad_mode="constant").abs()
            feat=torch.log1p(10*spec);feat=torch.cat([feat.mean(-1),feat.std(-1)],-1)
        return self.net(feat)

class NoiseAdaptiveResidualDemucs(nn.Module):
    """Exact Demucs forward plus continuous FiLM and learned residual strength."""
    def __init__(self,demucs,embedding=128,film_scale=.10,strength_range=.15):
        super().__init__();self.base=demucs;self.conditioner=SpectralNoiseConditioner(embedding=embedding)
        bottleneck=int(self.base.encoder[-1][0].out_channels)
        self.film=nn.Linear(embedding,2*bottleneck)
        self.strength=nn.Linear(embedding,1)
        nn.init.zeros_(self.film.weight);nn.init.zeros_(self.film.bias)
        nn.init.zeros_(self.strength.weight);nn.init.zeros_(self.strength.bias)
        self.film_scale=float(film_scale);self.strength_range=float(strength_range)
    def forward(self,mix,return_aux=False):
        if mix.dim()==2:mix=mix.unsqueeze(1)
        original=mix;embedding=self.conditioner(original[:,0])
        if self.base.normalize:
            std=original.mean(1,keepdim=True).std(-1,keepdim=True);x=original/(self.base.floor+std)
        else:std=1.;x=original
        length=x.shape[-1];x=F.pad(x,(0,self.base.valid_length(length)-length))
        if self.base.resample==2:x=upsample2(x)
        elif self.base.resample==4:x=upsample2(upsample2(x))
        skips=[]
        for encode in self.base.encoder:x=encode(x);skips.append(x)
        x=x.permute(2,0,1);x,_=self.base.lstm(x);x=x.permute(1,2,0)
        gamma,beta=self.film(embedding).chunk(2,-1)
        x=x*(1+self.film_scale*torch.tanh(gamma)[:,:,None])+self.film_scale*torch.tanh(beta)[:,:,None]
        for decode in self.base.decoder:
            skip=skips.pop(-1);x=decode(x+skip[...,:x.shape[-1]])
        if self.base.resample==2:x=downsample2(x)
        elif self.base.resample==4:x=downsample2(downsample2(x))
        speech_candidate=std*x[...,:length]
        strength=1+self.strength_range*torch.tanh(self.strength(embedding))[:,:,None]
        # Stable algebra: when strength==1 this is exactly speech_candidate.
        residual=original-speech_candidate
        predicted_noise=strength*residual
        enhanced=speech_candidate+(1-strength)*residual
        return (enhanced,predicted_noise,strength,embedding) if return_aux else enhanced


model=NoiseAdaptiveResidualDemucs(base)
source_state=torch.load(SOURCE_CHECKPOINT,map_location="cpu",weights_only=True)
missing,unexpected=model.load_state_dict(source_state,strict=False)
if missing or unexpected:
    raise RuntimeError(f"Source checkpoint mismatch: missing={missing[:8]} unexpected={unexpected[:8]}")
SOURCE_SHA=sha_file(SOURCE_CHECKPOINT)
with torch.no_grad():
    probe=torch.randn(2,1,16000)
    actual=model(probe)
if actual.shape!=probe.shape or not torch.isfinite(actual).all():
    raise RuntimeError("Loaded V34 model failed waveform contract")
print("SOURCE CHECKPOINT LOAD PASS |",SOURCE_CHECKPOINT,"| SHA",SOURCE_SHA)


SOURCE CHECKPOINT LOAD PASS | /content/drive/MyDrive/PHD/Datathon/track2_phase2_v35_asr_constrained_adaptive_demucs/run_v1/asrft_epoch_03.pt | SHA 9288e94a01817088a01811ca11b112b3732753988206c4eccf92231f32a2d820


## 5 — Teacher-distillation and dynamic-remixing fine-tuning

True released noisy/clean pairs and changed Phase-1 teacher pairs form the
speech and residual pools. Each epoch produces new cross-pair mixtures with
random noise count, offset, polarity and SNR. Unchanged Phase-1 natural audio
never becomes a clean target.


In [13]:
def neg_si_sdr_items(y,s):
    y=y-y.mean(-1,keepdim=True);s=s-s.mean(-1,keepdim=True)
    target=(y*s).sum(-1,keepdim=True)/(s.square().sum(-1,keepdim=True)+1e-8)*s
    error=y-target
    return -10*torch.log10((target.square().sum(-1)+1e-8)/(error.square().sum(-1)+1e-8))/20

def event_dicts(raw):
    if raw is None:return []
    if isinstance(raw,str):
        raw=raw.strip()
        if not raw:return []
        raw=json.loads(raw)
    if isinstance(raw,np.ndarray):raw=raw.tolist()
    out=[]
    for e in raw if isinstance(raw,(list,tuple)) else []:
        if not isinstance(e,dict):continue
        a=e.get("start",e.get("onset"));b=e.get("end",e.get("offset"))
        if a is not None and b is not None:out.append(dict(start=float(a),end=float(b)))
    return out

gold_records=[dict(pair_id="gold_"+r["clip_id"],source="gold",noisy_path=r["noisy_path"],
                   clean_path=r["clean_path"],events=r["events"])
              for r in TRAIN_ROWS.to_dict("records")]
teacher_records=[dict(pair_id=r["pair_id"],source="teacher",noisy_path=r["noisy_path"],
                      clean_path=r["clean_path"],events=[])
                 for r in TEACHER.to_dict("records")]
all_records=gold_records+teacher_records
print("Gold pairs",len(gold_records),"| teacher pairs",len(teacher_records))

def crop_or_pad(y,start,length=SEGMENT):
    z=y[start:start+length]
    return np.pad(z,(0,max(0,length-len(z))))[:length].astype(np.float32)

def random_segment(y,rng,length=SEGMENT):
    start=int(rng.integers(0,len(y)-length+1)) if len(y)>length else 0
    return crop_or_pad(y,start,length)

def event_segment(x,s,rng):
    n=x-s;length=min(SEGMENT,len(x))
    if len(x)<=length:return crop_or_pad(x,0),crop_or_pad(s,0)
    frame=400;hop=160
    energy=np.convolve(n.astype(np.float64)**2,np.ones(frame)/frame,mode="same")
    active=np.flatnonzero(energy>np.quantile(energy,.75))
    anchor=int(rng.choice(active)) if len(active) else int(rng.integers(0,len(x)))
    start=max(0,min(len(x)-length,anchor-int(rng.integers(0,length+1))))
    return crop_or_pad(x,start),crop_or_pad(s,start)

class DynamicTeacherDataset(Dataset):
    def __init__(self,records,draws,training=True):self.records=list(records);self.draws=draws;self.training=training;self.epoch=0
    def set_epoch(self,epoch):self.epoch=int(epoch)
    def __len__(self):return self.draws
    def load(self,r):return read_wave_strict(r["noisy_path"]),read_wave_strict(r["clean_path"])
    def __getitem__(self,i):
        rng=np.random.default_rng(SEED+self.epoch*1000003+i*9176)
        u=float(rng.random())
        if u<IDENTITY_PROBABILITY:
            r=self.records[int(rng.integers(0,len(self.records)))];_,s=self.load(r);s=random_segment(s,rng)
            x=s.copy();n=np.zeros_like(s)
        elif u<IDENTITY_PROBABILITY+DIRECT_PAIR_PROB:
            # Preserve a substantial fraction of exact teacher/gold mappings.
            if rng.random()<.35:r=gold_records[int(rng.integers(0,len(gold_records)))]
            else:r=teacher_records[int(rng.integers(0,len(teacher_records)))]
            xx,ss=self.load(r);x,s=event_segment(xx,ss,rng);n=x-s
        else:
            speech_r=self.records[int(rng.integers(0,len(self.records)))];_,speech=self.load(speech_r)
            s=random_segment(speech,rng);n=np.zeros_like(s)
            count=int(rng.choice([1,2,3],p=[.55,.30,.15]))
            speech_rms=float(np.sqrt(np.mean(s.astype(np.float64)**2))+1e-6)
            for _ in range(count):
                noise_r=self.records[int(rng.integers(0,len(self.records)))];nx,ns=self.load(noise_r)
                source_noise=nx-ns;piece=random_segment(source_noise,rng)
                piece=np.roll(piece,int(rng.integers(0,SEGMENT)))
                if rng.random()<.5:piece=-piece
                noise_rms=float(np.sqrt(np.mean(piece.astype(np.float64)**2))+1e-7)
                snr=float(rng.uniform(-8,18));scale=speech_rms/(noise_rms*10**(snr/20))
                n+=piece*np.float32(scale/count**.5)
            x=s+n
            peak=float(max(np.max(np.abs(x)),np.max(np.abs(s)),1e-8))
            if peak>.98:x*=.98/peak;s*=.98/peak;n*=.98/peak
        # Soft activity mask is supervision only; inference never gates or bypasses.
        smooth=np.convolve(n.astype(np.float64)**2,np.ones(400)/400,mode="same")
        threshold=max(1e-8,float(np.quantile(smooth,.60))*.20)
        mask=(smooth>threshold).astype(np.float32)
        return tuple(torch.from_numpy(z.astype(np.float32)) for z in (x,s,n,mask))

train_ds=DynamicTeacherDataset(all_records,DRAWS_PER_EPOCH,True)
train_dl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True,
                    drop_last=True,persistent_workers=True)

def complex_noise_loss(pred,target):
    total=pred.new_zeros(())
    for size in (256,512,1024):
        win=torch.hann_window(size,device=pred.device)
        P=torch.stft(pred.float(),size,hop_length=size//4,window=win,return_complex=True,pad_mode="constant")
        T=torch.stft(target.float(),size,hop_length=size//4,window=win,return_complex=True,pad_mode="constant")
        total+=(P-T).abs().mean()/T.abs().mean().clamp_min(1e-3)
    return total/3

def distillation_loss(net,x,s,target_n,event_mask):
    y,pred_n,strength,_=net(x[:,None],return_aux=True);y=y[:,0];pred_n=pred_n[:,0]
    speech_si=neg_si_sdr_items(y,s).mean()
    valid=target_n.square().mean(-1)>1e-8
    noise_si=neg_si_sdr_items(pred_n[valid],target_n[valid]).mean() if valid.any() else y.new_zeros(())
    complex_n=complex_noise_loss(pred_n,target_n)
    scale=target_n.square().mean(-1).sqrt().clamp_min(.01)[:,None]
    noise_wave=((pred_n-target_n).abs()/scale).mean()
    outside=1-event_mask
    speech_scale=s.square().mean(-1).sqrt().clamp_min(.01)[:,None]
    outside_identity=(((y-x).abs()/speech_scale)*outside).sum()/outside.sum().clamp_min(1)
    loss=(speech_si+NOISE_SISDR_WEIGHT*noise_si+COMPLEX_NOISE_WEIGHT*complex_n+
          NOISE_WAVE_WEIGHT*noise_wave+OUTSIDE_IDENTITY_WEIGHT*outside_identity)
    return loss,dict(speech_si=speech_si,noise_si=noise_si,complex_noise=complex_n,
                     noise_wave=noise_wave,outside_identity=outside_identity,mean_strength=strength.mean())


Gold pairs 591 | teacher pairs 2155


In [14]:
def flatten_rnns(net):
    for module in net.modules():
        if isinstance(module,nn.RNNBase):module.flatten_parameters()

history_path=RUN/"teacher_distillation_history.csv"
history=pd.read_csv(history_path).to_dict("records") if history_path.exists() else []
existing=sorted(RUN.glob("teacher_epoch_*.pt"));start_epoch=0
if existing:
    latest=existing[-1];model.load_state_dict(torch.load(latest,map_location="cpu",weights_only=True),strict=True)
    start_epoch=int(latest.stem.rsplit("_",1)[-1]);print("Resuming after epoch",start_epoch)
else:model.load_state_dict(source_state,strict=True)

def configure_optimizer(epoch):
    for p in model.parameters():p.requires_grad_(False)
    adapters=(model.conditioner,model.film,model.strength)
    for module in adapters:
        for p in module.parameters():p.requires_grad_(True)
    for p in model.base.decoder.parameters():p.requires_grad_(True)
    groups=[dict(params=[p for m in adapters for p in m.parameters()],lr=LR_ADAPTER),
            dict(params=model.base.decoder.parameters(),lr=LR_DECODER)]
    stage="adapter_decoder"
    if epoch>=3:
        for p in model.base.encoder.parameters():p.requires_grad_(True)
        for p in model.base.lstm.parameters():p.requires_grad_(True)
        groups.append(dict(params=list(model.base.encoder.parameters())+list(model.base.lstm.parameters()),lr=LR_FULL))
        stage="full_low_lr"
    return torch.optim.AdamW(groups,weight_decay=1e-5),stage

model.to(DEVICE);flatten_rnns(model)
# Short learning/gradient check before committing to the full run.
probe_batch=next(iter(train_dl));px,ps,pn,pm=[v[:2].to(DEVICE) for v in probe_batch]
optimizer,stage=configure_optimizer(1);optimizer.zero_grad(set_to_none=True)
with torch.amp.autocast("cuda",enabled=True):probe_loss,probe_parts=distillation_loss(model,px,ps,pn,pm)
probe_loss.backward()
checks={"conditioner":any(p.grad is not None and p.grad.abs().sum()>0 for p in model.conditioner.parameters()),
        "decoder":any(p.grad is not None and p.grad.abs().sum()>0 for p in model.base.decoder.parameters())}
optimizer.zero_grad(set_to_none=True)
if not all(checks.values()) or not torch.isfinite(probe_loss):raise RuntimeError(f"Learning check failed {checks}")
print("DISTILLATION GRADIENT CHECK PASS",checks,{k:float(v.detach()) for k,v in probe_parts.items()})
del px,ps,pn,pm,probe_batch,probe_loss,probe_parts,optimizer

CANDIDATES={"v35_source":SOURCE_CHECKPOINT}
for epoch in range(start_epoch+1,FINETUNE_EPOCHS+1):
    train_ds.set_epoch(epoch);optimizer,stage=configure_optimizer(epoch)
    model.train();flatten_rnns(model);losses=[];components=[]
    for x,s,n,m in tqdm(train_dl,desc=f"Teacher distillation {epoch}/{FINETUNE_EPOCHS}"):
        x=x.to(DEVICE,non_blocking=True);s=s.to(DEVICE,non_blocking=True)
        n=n.to(DEVICE,non_blocking=True);m=m.to(DEVICE,non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda",enabled=True):loss,parts=distillation_loss(model,x,s,n,m)
        if not torch.isfinite(loss):raise RuntimeError("Non-finite training loss")
        loss.backward();trainable=[p for p in model.parameters() if p.requires_grad]
        nn.utils.clip_grad_norm_(trainable,3.,error_if_nonfinite=True);optimizer.step()
        losses.append(float(loss.detach()));components.append({k:float(v.detach()) for k,v in parts.items()})
    means=pd.DataFrame(components).mean().to_dict();rec=dict(epoch=epoch,stage=stage,
        train_loss=float(np.mean(losses)),**means)
    history=[r for r in history if int(r["epoch"])!=epoch]+[rec];history=sorted(history,key=lambda r:int(r["epoch"]))
    path=RUN/f"teacher_epoch_{epoch:02d}.pt";tmp=path.with_suffix(".tmp")
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},tmp);os.replace(tmp,path)
    pd.DataFrame(history).to_csv(history_path,index=False);print(rec);del optimizer

for p in sorted(RUN.glob("teacher_epoch_*.pt")):CANDIDATES[p.stem]=p
model.cpu();gc.collect();torch.cuda.empty_cache();print("Candidates",list(CANDIDATES))


DISTILLATION GRADIENT CHECK PASS {'conditioner': True, 'decoder': True} {'speech_si': -0.7388636469841003, 'noise_si': -0.5944247841835022, 'complex_noise': 0.24024221301078796, 'noise_wave': 0.11712457239627838, 'outside_identity': 0.009165413677692413, 'mean_strength': 1.0}


Teacher distillation 1/4:   0%|          | 0/1000 [00:00<?, ?it/s]

{'epoch': 1, 'stage': 'adapter_decoder', 'train_loss': -0.8684470767863095, 'speech_si': -0.7951648604013026, 'noise_si': -0.5631327518858016, 'complex_noise': 0.2901386389359832, 'noise_wave': 0.1514390100799501, 'outside_identity': 0.03989528299076483, 'mean_strength': 0.9987265625}


Teacher distillation 2/4:   0%|          | 0/1000 [00:00<?, ?it/s]

{'epoch': 2, 'stage': 'adapter_decoder', 'train_loss': -0.8722776844836772, 'speech_si': -0.7980626105852425, 'noise_si': -0.565149393260479, 'complex_noise': 0.2893662741109729, 'noise_wave': 0.1508387564253062, 'outside_identity': 0.039521532038925215, 'mean_strength': 0.9986865234375}


Teacher distillation 3/4:   0%|          | 0/1000 [00:00<?, ?it/s]

{'epoch': 3, 'stage': 'full_low_lr', 'train_loss': -0.8751294321008026, 'speech_si': -0.8002566561028361, 'noise_si': -0.5669301128201186, 'complex_noise': 0.28899966675788163, 'noise_wave': 0.15049377653747797, 'outside_identity': 0.03953485503583215, 'mean_strength': 0.99872705078125}


Teacher distillation 4/4:   0%|          | 0/1000 [00:00<?, ?it/s]

{'epoch': 4, 'stage': 'full_low_lr', 'train_loss': -0.8783245728760958, 'speech_si': -0.802636513914913, 'noise_si': -0.5688408031500876, 'complex_noise': 0.288278868496418, 'noise_wave': 0.15003061073645949, 'outside_identity': 0.03945705187658314, 'mean_strength': 0.99871875}
Candidates ['v35_source', 'teacher_epoch_01', 'teacher_epoch_02', 'teacher_epoch_03', 'teacher_epoch_04']


## 6 — Sign-ensembled, always-on inference and frozen SraVaani evaluation


In [15]:
LAST_MODEL_CALLS=0
@torch.inference_mode()
def enhance_audio(y):
    global LAST_MODEL_CALLS
    y=np.asarray(y,np.float32);out=np.zeros(len(y),np.float64);den=np.zeros(len(y),np.float64)
    weight=np.maximum(np.hanning(CHUNK_SAMPLES),1e-3);LAST_MODEL_CALLS=0
    for start in range(0,len(y),CHUNK_HOP):
        count=min(CHUNK_SAMPLES,len(y)-start);seg=np.pad(y[start:start+count],(0,CHUNK_SAMPLES-count)).astype(np.float32)
        xt=torch.from_numpy(seg)[None].to(DEVICE)
        _,n1,_,_=model(xt[:,None],return_aux=True)
        if SIGN_ENSEMBLE:
            _,n2,_,_=model(-xt[:,None],return_aux=True);pred_n=.5*(n1-n2);LAST_MODEL_CALLS+=2
        else:pred_n=n1;LAST_MODEL_CALLS+=1
        z=(xt[:,None]-float(GLOBAL_ALPHA)*pred_n)[0,0].float().cpu().numpy()
        if z.shape!=(CHUNK_SAMPLES,) or not np.isfinite(z).all():raise RuntimeError("Invalid enhancement; no fallback")
        out[start:start+count]+=z[:count]*weight[:count];den[start:start+count]+=weight[:count]
        if start+CHUNK_SAMPLES>=len(y):break
    if (den<=0).any():raise RuntimeError("Uncovered samples")
    return (out/den).astype(np.float32)

def activate(path,device=DEVICE):
    try:state=torch.load(path,map_location="cpu",weights_only=True)
    except TypeError:state=torch.load(path,map_location="cpu")
    model.load_state_dict(state,strict=True);model.to(device).eval();flatten_rnns(model)


In [18]:
import importlib
from transformers import AutoModel
pin=json.loads(ASR_PIN.read_text()); ASR_REV=str(pin['revision'])
if pin.get('repo')!=ASR_REPO or not re.fullmatch('[0-9a-fA-F]{40}',ASR_REV):
    raise ValueError('Invalid existing SraVaani revision pin')
asr_identity=dict(repo=ASR_REPO,revision=ASR_REV,task='transcribe_return_hypotheses',
    pipeline='v29_content_filename_nograd_v1',torch=str(torch.__version__),
    transformers=importlib.metadata.version('transformers'))
ASR_SIGNATURE=sha_json(asr_identity)
token=os.environ.get('HF_TOKEN')
if not token:
    try:
        from google.colab import userdata
        token=userdata.get('HF_TOKEN')
    except Exception:pass
asr_model=AutoModel.from_pretrained(ASR_REPO,revision=ASR_REV,code_revision=ASR_REV,
    trust_remote_code=True,token=token).to(DEVICE).eval()
put_json(RUN/'asr_identity.json',asr_identity)

class FreshASR:
    def __init__(self,model):
        self.model=model;self.cache=ROOT/'track2_phase2_v28_pretrained_wer'/'asr_cache_v28'/ASR_SIGNATURE
        self.cache.mkdir(parents=True,exist_ok=True)
        self.stage=WORK/'asr_inputs';self.stage.mkdir(exist_ok=True)
    def transcribe(self,paths,force=False):
        result={}
        for cid,p in tqdm(paths.items(),desc='Content-audited SraVaani'):
            p=Path(p);h=sha_file(p);dest=self.cache/(h+'.json')
            rec=None
            if dest.exists() and not force:
                trial=json.loads(dest.read_text())
                if trial.get('wav_sha256')==h and trial.get('model_signature')==ASR_SIGNATURE:
                    rec=trial
            if rec is None:
                staged=self.stage/(h+'.wav')
                if not staged.exists() or sha_file(staged)!=h:shutil.copyfile(p,staged)
                if sha_file(staged)!=h:raise IOError('ASR input hash mismatch')
                # One audio per call makes result-to-file association unambiguous.
                with torch.inference_mode(False), torch.no_grad():
                    hyp=self.model.transcribe([str(staged)],return_hypotheses=True)
                text=hyp_text(hyp)
                rec=dict(text=text,wav_sha256=h,model_signature=ASR_SIGNATURE,
                         staged_name=staged.name,source_path=str(p))
                put_json(dest,rec)
            if not isinstance(rec.get('text'),str):raise ValueError('Invalid cached transcript')
            result[cid]=rec
        return result
ASR=FreshASR(asr_model)

def text_tokens(text):
    text=re.sub(r'<[^>]*>|\\[[^\\]]*\\]',' ',text)
    text=unicodedata.normalize('NFC',text).lower()
    return ''.join(' ' if unicodedata.category(c).startswith('P') else c for c in text).split()

def word_counts(ref,hyp):
    # Deterministic minimum-edit alignment; ties prefer substitution, then deletion.
    a,b=text_tokens(ref),text_tokens(hyp);n,m=len(a),len(b)
    dp=np.zeros((n+1,m+1),dtype=np.int32)
    dp[:,0]=np.arange(n+1);dp[0,:]=np.arange(m+1)
    for i in range(1,n+1):
        for j in range(1,m+1):
            dp[i,j]=min(dp[i-1,j-1]+(a[i-1]!=b[j-1]),dp[i-1,j]+1,dp[i,j-1]+1)
    i,j=n,m;sub=delete=insert=hit=0
    while i or j:
        if i and j and a[i-1]==b[j-1] and dp[i,j]==dp[i-1,j-1]:hit+=1;i-=1;j-=1
        elif i and j and dp[i,j]==dp[i-1,j-1]+1:sub+=1;i-=1;j-=1
        elif i and dp[i,j]==dp[i-1,j]+1:delete+=1;i-=1
        else:insert+=1;j-=1
    return dict(words=n,substitutions=sub,deletions=delete,insertions=insert,
                errors=sub+delete+insert,hits=hit)
assert word_counts('a b','a c')['substitutions']==1
assert word_counts('a b','a')['deletions']==1
assert word_counts('a','a b')['insertions']==1

BASELINES={}
def get_reference_decodes(label,rows):
    if label in BASELINES:return BASELINES[label]
    raw={r.clip_id:Path(r.noisy_path) for r in rows.itertuples()}
    raw_h=ASR.transcribe(raw)
    clean_h=ASR.transcribe({r.clip_id:Path(r.clean_path) for r in rows.itertuples()}) if 'clean_path' in rows else None
    BASELINES[label]=(raw_h,clean_h)
    return raw_h,clean_h
get_reference_decodes('synthetic_dev',SYN_DEV)
get_reference_decodes('natural_dev',NAT_DEV)
# Independent uncached re-decode of one noisy/clean pair.
first=SYN_DEV.iloc[0]
check=ASR.transcribe({'noisy':Path(first.noisy_path),'clean':Path(first.clean_path)},force=True)
put_json(RUN/'uncached_pair_check.json',check)
print('First pair fresh decode:',{k:v['text'] for k,v in check.items()})

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Content-audited SraVaani:   0%|          | 0/104 [00:00<?, ?it/s]

Content-audited SraVaani:   0%|          | 0/104 [00:00<?, ?it/s]

Content-audited SraVaani:   0%|          | 0/100 [00:00<?, ?it/s]

Content-audited SraVaani:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:1790: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1479.)
  return forward_call(*args, **kwargs)


First pair fresh decode: {'noisy': 'ধাবাখনৰ বাহিৰত এখন ৰঙা ৰঙৰ গাড়ী দেখিবলৈ পাইছোঁ', 'clean': 'ধাবাখনৰ বাহিৰত এখন ৰঙা ৰঙৰ গাড়ী দেখিবলৈ পাইছোঁ'}


In [19]:
def process_rows(rows,directory,checkpoint_sha):
    directory.mkdir(parents=True,exist_ok=True);paths={};audit=[]
    for r in tqdm(rows.to_dict('records'),desc='Always-on enhancement'):
        cid=r['clip_id'];x=read_wave_strict(r['noisy_path'])
        p=directory/(cid+'.wav');rec_path=directory/(cid+'.audit.json')
        key=sha_json(dict(input=hashlib.sha256(x.tobytes()).hexdigest(),checkpoint=checkpoint_sha,impl=IMPLEMENTATION,global_alpha=float(GLOBAL_ALPHA),sign_ensemble=bool(SIGN_ENSEMBLE)))
        rec=None
        if p.exists() and rec_path.exists():
            trial=json.loads(rec_path.read_text())
            if trial.get('key')==key and trial.get('wav_sha256')==wav_audit(p,len(x)):rec=trial
        if rec is None:
            z=enhance_audio(x);write_pcm(p,z)
            rec=dict(clip_id=cid,key=key,model_calls=LAST_MODEL_CALLS,wav_sha256=wav_audit(p,len(x)),
                unchanged_pcm=bool(np.array_equal(canonical_pcm(x),canonical_pcm(z))),
                clipped_samples=int(np.count_nonzero((z<-1)|(z>32767/32768))),
                change_rms=float(np.sqrt(np.mean((pcm_roundtrip(z)-pcm_roundtrip(x))**2))))
            put_json(rec_path,rec)
        paths[cid]=p;audit.append(rec)
    return paths,audit

def evaluate(label,rows,paths,candidate):
    raw,clean=get_reference_decodes(label,rows)
    enh=ASR.transcribe(paths);details=[];skipped=0
    for r in rows.to_dict('records'):
        cid=r['clip_id'];ref=r['transcript']
        if not text_tokens(ref):skipped+=1;continue
        record=dict(clip_id=cid,reference=ref,raw_text=raw[cid]['text'],enhanced_text=enh[cid]['text'],
            raw_sha=raw[cid]['wav_sha256'],enhanced_sha=enh[cid]['wav_sha256'],
            same_transcript=text_tokens(raw[cid]['text'])==text_tokens(enh[cid]['text']))
        for role,h in [('raw',raw),('enhanced',enh),('clean',clean)]:
            if h is not None:
                record.update({role+'_'+k:v for k,v in word_counts(ref,h[cid]['text']).items()})
                if role=='clean':record['clean_text']=h[cid]['text'];record['clean_sha']=h[cid]['wav_sha256']
        if 'clean_path' in r:
            s=read_wave_strict(r['clean_path']);x=read_wave_strict(r['noisy_path']);z=read_wave_strict(paths[cid])
            record['noisy_si_sdr']=si_sdr(x,s);record['enhanced_si_sdr']=si_sdr(z,s)
        record['error_increase']=record['enhanced_errors']-record['raw_errors']
        details.append(record)
    if not details:raise ValueError('No usable reference transcripts')
    frame=pd.DataFrame(details).sort_values('error_increase',ascending=False)
    frame.to_csv(RUN/(candidate+'_'+label+'_details.csv'),index=False)
    words=int(frame.raw_words.sum());summary=dict(candidate=candidate,subset=label,clips=len(frame),skipped=skipped,reference_words=words,raw_errors=int(frame.raw_errors.sum()),enhanced_errors=int(frame.enhanced_errors.sum()),
        raw_wer_percent=100*frame.raw_errors.sum()/words,
        enhanced_wer_percent=100*frame.enhanced_errors.sum()/words,
        identical_transcript_fraction=float(frame.same_transcript.mean()))
    summary['delta_wer_points']=summary['raw_wer_percent']-summary['enhanced_wer_percent']
    for role in ['raw','enhanced']:
        for typ in ['substitutions','deletions','insertions']:summary[role+'_'+typ]=int(frame[role+'_'+typ].sum())
    if clean is not None:
        summary['clean_wer_percent']=100*frame.clean_errors.sum()/words
        summary['enhanced_si_sdr']=float(frame.enhanced_si_sdr.mean())
        summary['noisy_si_sdr']=float(frame.noisy_si_sdr.mean())
    print(summary);return summary


In [20]:
summaries=[];selection=[]
for candidate,checkpoint in CANDIDATES.items():
    asr_model.to("cpu");torch.cuda.empty_cache();activate(checkpoint);checkpoint_sha=sha_file(checkpoint)
    syn_paths,syn_audit=process_rows(SYN_DEV,WORK/candidate/"synthetic_dev",checkpoint_sha)
    nat_paths,nat_audit=process_rows(NAT_DEV,WORK/candidate/"natural_dev",checkpoint_sha)
    model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval()
    a=evaluate("synthetic_dev",SYN_DEV,syn_paths,candidate);b=evaluate("natural_dev",NAT_DEV,nat_paths,candidate)
    summaries.extend([a,b]);words=a["reference_words"]+b["reference_words"]
    dwer=100*((a["raw_errors"]+b["raw_errors"])-(a["enhanced_errors"]+b["enhanced_errors"]))/words
    proxy=a["enhanced_si_sdr"]+dwer
    unchanged=any(r["unchanged_pcm"] for r in syn_audit+nat_audit)
    selection.append(dict(candidate=candidate,synthetic_si_sdr=a["enhanced_si_sdr"],pooled_delta_wer_points=dwer,
                          combined_proxy=proxy,unchanged=unchanged))
    pd.DataFrame(syn_audit+nat_audit).to_csv(RUN/(candidate+"_processing_audit.csv"),index=False)
    put_json(RUN/"development_metrics.json",summaries)
eligible_sel=[r for r in selection if not r["unchanged"]]
if not eligible_sel:raise RuntimeError("No genuinely processing checkpoint")
BEST=max(eligible_sel,key=lambda r:r["combined_proxy"])["candidate"]
SELECTED=CANDIDATES[BEST];SELECTED_SHA=sha_file(SELECTED)
put_json(RUN/"selected_model.json",dict(candidate=BEST,checkpoint=str(SELECTED),sha256=SELECTED_SHA,scores=selection,
         criterion="synthetic SI-SDR plus pooled WER improvement points",global_alpha=GLOBAL_ALPHA,sign_ensemble=SIGN_ENSEMBLE))
display(pd.DataFrame(selection).sort_values("combined_proxy",ascending=False));print("Selected:",BEST)

# Monitoring is reported once and never used to reselect among candidates.
asr_model.to("cpu");torch.cuda.empty_cache();activate(SELECTED)
sp,sa=process_rows(SYN_MON,WORK/BEST/"synthetic_monitor",SELECTED_SHA)
npth,na=process_rows(NAT_MON,WORK/BEST/"natural_monitor",SELECTED_SHA)
model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval()
monitor=[evaluate("synthetic_monitor",SYN_MON,sp,BEST),evaluate("natural_monitor",NAT_MON,npth,BEST)]
put_json(RUN/"monitor_metrics.json",monitor)


Always-on enhancement:   0%|          | 0/104 [00:00<?, ?it/s]

Always-on enhancement:   0%|          | 0/100 [00:00<?, ?it/s]

Content-audited SraVaani:   0%|          | 0/104 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:1790: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1479.)
  return forward_call(*args, **kwargs)


{'candidate': 'v35_source', 'subset': 'synthetic_dev', 'clips': 104, 'skipped': 0, 'reference_words': 1321, 'raw_errors': 458, 'enhanced_errors': 457, 'raw_wer_percent': np.float64(34.67070401211203), 'enhanced_wer_percent': np.float64(34.59500378501136), 'identical_transcript_fraction': 0.34615384615384615, 'delta_wer_points': np.float64(0.07570022710067548), 'raw_substitutions': 217, 'raw_deletions': 234, 'raw_insertions': 7, 'enhanced_substitutions': 237, 'enhanced_deletions': 213, 'enhanced_insertions': 7, 'clean_wer_percent': np.float64(22.937168811506435), 'enhanced_si_sdr': 18.476780293459647, 'noisy_si_sdr': 0.41693136310790574}


Content-audited SraVaani:   0%|          | 0/100 [00:00<?, ?it/s]

... (43 outputs left out)

Content-audited SraVaani:   0%|          | 0/100 [00:00<?, ?it/s]

Content-audited SraVaani:   0%|          | 0/100 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:1790: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1479.)
  return forward_call(*args, **kwargs)


{'candidate': 'teacher_epoch_01', 'subset': 'natural_monitor', 'clips': 100, 'skipped': 0, 'reference_words': 1468, 'raw_errors': 480, 'enhanced_errors': 477, 'raw_wer_percent': np.float64(32.69754768392371), 'enhanced_wer_percent': np.float64(32.49318801089918), 'identical_transcript_fraction': 0.79, 'delta_wer_points': np.float64(0.20435967302452696), 'raw_substitutions': 225, 'raw_deletions': 245, 'raw_insertions': 10, 'enhanced_substitutions': 221, 'enhanced_deletions': 246, 'enhanced_insertions': 10}


## 7 — Generate the selected Phase-2 submission ZIP


In [21]:
if RUN_PHASE2:
    if not CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING:raise RuntimeError("Rules confirmation required")
    ids=sorted(TRANSCRIPT_MAP)
    if len(ids)!=EXPECTED_PHASE2_CLIPS:raise ValueError("Unexpected Phase-2 clip count")
    source=StrictWaveSource(SOURCE_AUDIO,ids)
    asr_model.to("cpu");torch.cuda.empty_cache();activate(SELECTED)
    wavs=WORK/"phase2"/SELECTED_SHA[:12];wavs.mkdir(parents=True,exist_ok=True)
    paths={};frames={};audits=[]
    for cid in tqdm(ids,desc="All-clip always-on noise-adaptive enhancement"):
        x=source.load(cid);z=enhance_audio(x);p=wavs/(cid+".wav");write_pcm(p,z)
        audits.append(dict(clip_id=cid,input_sha256=hashlib.sha256(x.tobytes()).hexdigest(),
            wav_sha256=wav_audit(p,len(x)),checkpoint_sha256=SELECTED_SHA,model_calls=LAST_MODEL_CALLS,
            unchanged_pcm=bool(np.array_equal(canonical_pcm(x),canonical_pcm(z))),
            clipped_samples=int(np.count_nonzero((z<-1)|(z>32767/32768)))))
        paths[cid]=p;frames[cid]=len(x)
    source.close();pd.DataFrame(audits).to_csv(RUN/"phase2_processing_audit.csv",index=False)
    if any(r["unchanged_pcm"] for r in audits):raise RuntimeError("Unchanged output found; no bypass permitted")
    model.to("cpu");torch.cuda.empty_cache();asr_model.to(DEVICE).eval();transcripts=ASR.transcribe(paths)
    ZIP=RUN/("submission_Track2_PHASE2_V36_Phase1Teacher_"+BEST+"_"+SELECTED_SHA[:10]+".zip")
    package=pack_submission(paths,transcripts,frames,ZIP)
    put_json(RUN/"submission_manifest.json",dict(version="v36_phase1_dictionary_teacher_demucs",candidate=BEST,
        checkpoint_sha256=SELECTED_SHA,baseline_checkpoint_sha256=BASELINE_SHA,source_checkpoint_sha256=SOURCE_SHA,all_clips_enhanced=True,
        routing=False,dictionary_inference=False,classifier_routing=False,beats_in_inference=False,
        tts_in_inference=False,continuous_noise_conditioning=True,residual_subtraction=True,phase1_dictionary_teacher_training=True,global_alpha=GLOBAL_ALPHA,
        sign_ensemble=SIGN_ENSEMBLE,teacher_pairs=int(len(TEACHER)),
        source_derived_training_permission_confirmed=True,package=package))
    print("ZIP ready for manual upload:",ZIP)
else:print("RUN_PHASE2=False: metrics generated; no submission ZIP created.")


All-clip always-on noise-adaptive enhancement:   0%|          | 0/3178 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [22]:
# Fixed global removal-strength sweep for teacher_epoch_01.
# Selection uses SYN_DEV + NAT_DEV only. Monitoring is evaluated once afterward.

SWEEP_CHECKPOINT = RUN / "teacher_epoch_01.pt"

if not SWEEP_CHECKPOINT.is_file():
    raise FileNotFoundError(SWEEP_CHECKPOINT)

ALPHA_VALUES = [
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.95,
    1.00,
]

SWEEP_ROOT = WORK / "teacher_epoch01_alpha_sweep"
SWEEP_ROOT.mkdir(parents=True, exist_ok=True)

SWEEP_SHA = sha_file(SWEEP_CHECKPOINT)
sweep_rows = []

# Decode raw reference files once before the sweep.
asr_model.to(DEVICE).eval()
get_reference_decodes("synthetic_dev", SYN_DEV)
get_reference_decodes("natural_dev", NAT_DEV)

for alpha in ALPHA_VALUES:
    GLOBAL_ALPHA = float(alpha)
    alpha_tag = f"alpha_{int(round(alpha * 100)):03d}"
    candidate = f"teacher_epoch01_{alpha_tag}"

    print("\n" + "=" * 80)
    print("Evaluating:", candidate)
    print("GLOBAL_ALPHA:", GLOBAL_ALPHA)

    # Enhancement stage
    asr_model.to("cpu")
    gc.collect()
    torch.cuda.empty_cache()

    activate(SWEEP_CHECKPOINT)

    syn_paths, syn_audit = process_rows(
        SYN_DEV,
        SWEEP_ROOT / alpha_tag / "synthetic_dev",
        SWEEP_SHA,
    )

    nat_paths, nat_audit = process_rows(
        NAT_DEV,
        SWEEP_ROOT / alpha_tag / "natural_dev",
        SWEEP_SHA,
    )

    # ASR evaluation stage
    model.to("cpu")
    gc.collect()
    torch.cuda.empty_cache()

    asr_model.to(DEVICE).eval()

    syn = evaluate(
        "synthetic_dev",
        SYN_DEV,
        syn_paths,
        candidate,
    )

    nat = evaluate(
        "natural_dev",
        NAT_DEV,
        nat_paths,
        candidate,
    )

    total_words = (
        syn["reference_words"]
        + nat["reference_words"]
    )

    raw_errors = (
        syn["raw_errors"]
        + nat["raw_errors"]
    )

    enhanced_errors = (
        syn["enhanced_errors"]
        + nat["enhanced_errors"]
    )

    pooled_delta_wer = (
        100.0
        * (raw_errors - enhanced_errors)
        / total_words
    )

    synthetic_deletion_change = (
        syn["enhanced_deletions"]
        - syn["raw_deletions"]
    )

    natural_deletion_change = (
        nat["enhanced_deletions"]
        - nat["raw_deletions"]
    )

    audit = syn_audit + nat_audit
    unchanged = any(
        bool(record["unchanged_pcm"])
        for record in audit
    )

    mean_change_rms = float(
        np.mean(
            [
                record["change_rms"]
                for record in audit
            ]
        )
    )

    combined_proxy = (
        syn["enhanced_si_sdr"]
        + pooled_delta_wer
    )

    # Development-only safety guard.
    # This rejects aggressive settings that obtain SI-SDR by increasing
    # deletions or significantly damaging natural-audio recognition.
    eligible = bool(
        not unchanged
        and pooled_delta_wer >= -0.30
        and nat["delta_wer_points"] >= -0.50
        and synthetic_deletion_change <= 10
        and natural_deletion_change <= 6
    )

    sweep_rows.append(
        {
            "candidate": candidate,
            "global_alpha": GLOBAL_ALPHA,
            "synthetic_si_sdr":
                syn["enhanced_si_sdr"],
            "synthetic_delta_wer_points":
                syn["delta_wer_points"],
            "natural_delta_wer_points":
                nat["delta_wer_points"],
            "pooled_delta_wer_points":
                pooled_delta_wer,
            "combined_proxy":
                combined_proxy,
            "synthetic_raw_deletions":
                syn["raw_deletions"],
            "synthetic_enhanced_deletions":
                syn["enhanced_deletions"],
            "synthetic_deletion_change":
                synthetic_deletion_change,
            "natural_raw_deletions":
                nat["raw_deletions"],
            "natural_enhanced_deletions":
                nat["enhanced_deletions"],
            "natural_deletion_change":
                natural_deletion_change,
            "mean_change_rms":
                mean_change_rms,
            "unchanged":
                unchanged,
            "eligible":
                eligible,
        }
    )

    pd.DataFrame(audit).to_csv(
        RUN / f"{candidate}_processing_audit.csv",
        index=False,
    )


SWEEP_RESULTS = pd.DataFrame(sweep_rows)

SWEEP_RESULTS.to_csv(
    RUN / "teacher_epoch01_alpha_sweep.csv",
    index=False,
)

display(
    SWEEP_RESULTS.sort_values(
        [
            "eligible",
            "combined_proxy",
        ],
        ascending=[
            False,
            False,
        ],
    ).reset_index(drop=True)
)


eligible_rows = SWEEP_RESULTS[
    SWEEP_RESULTS["eligible"]
].copy()

if len(eligible_rows):
    selected_row = eligible_rows.sort_values(
        "combined_proxy",
        ascending=False,
    ).iloc[0]
else:
    print(
        "WARNING: no alpha passed every safety guard. "
        "Selecting the least damaging development setting."
    )

    selected_row = SWEEP_RESULTS.sort_values(
        [
            "pooled_delta_wer_points",
            "synthetic_deletion_change",
            "combined_proxy",
        ],
        ascending=[
            False,
            True,
            False,
        ],
    ).iloc[0]


SELECTED_ALPHA = float(
    selected_row["global_alpha"]
)

GLOBAL_ALPHA = SELECTED_ALPHA
SELECTED = SWEEP_CHECKPOINT
SELECTED_SHA = SWEEP_SHA
BEST = str(selected_row["candidate"])

print("\n" + "=" * 80)
print("Selected development alpha:", SELECTED_ALPHA)
print("Selected candidate:", BEST)
print(
    "Development combined proxy:",
    float(selected_row["combined_proxy"]),
)


# Independent monitoring: reported once and not used to select alpha.
asr_model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

activate(SELECTED)

monitor_tag = (
    f"alpha_{int(round(SELECTED_ALPHA * 100)):03d}"
)

monitor_syn_paths, monitor_syn_audit = process_rows(
    SYN_MON,
    SWEEP_ROOT
    / monitor_tag
    / "synthetic_monitor",
    SELECTED_SHA,
)

monitor_nat_paths, monitor_nat_audit = process_rows(
    NAT_MON,
    SWEEP_ROOT
    / monitor_tag
    / "natural_monitor",
    SELECTED_SHA,
)

model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

asr_model.to(DEVICE).eval()

monitor_syn = evaluate(
    "synthetic_monitor",
    SYN_MON,
    monitor_syn_paths,
    BEST,
)

monitor_nat = evaluate(
    "natural_monitor",
    NAT_MON,
    monitor_nat_paths,
    BEST,
)

monitor_words = (
    monitor_syn["reference_words"]
    + monitor_nat["reference_words"]
)

monitor_delta_wer = (
    100.0
    * (
        monitor_syn["raw_errors"]
        + monitor_nat["raw_errors"]
        - monitor_syn["enhanced_errors"]
        - monitor_nat["enhanced_errors"]
    )
    / monitor_words
)

MONITOR_SUMMARY = {
    "selected_alpha":
        SELECTED_ALPHA,
    "synthetic_si_sdr":
        monitor_syn["enhanced_si_sdr"],
    "synthetic_delta_wer_points":
        monitor_syn["delta_wer_points"],
    "natural_delta_wer_points":
        monitor_nat["delta_wer_points"],
    "pooled_delta_wer_points":
        monitor_delta_wer,
    "combined_proxy":
        monitor_syn["enhanced_si_sdr"]
        + monitor_delta_wer,
    "synthetic_deletion_change":
        monitor_syn["enhanced_deletions"]
        - monitor_syn["raw_deletions"],
    "natural_deletion_change":
        monitor_nat["enhanced_deletions"]
        - monitor_nat["raw_deletions"],
}

display(
    pd.DataFrame([MONITOR_SUMMARY])
)

put_json(
    RUN / "teacher_epoch01_selected_alpha.json",
    {
        "candidate": BEST,
        "checkpoint": str(SELECTED),
        "checkpoint_sha256": SELECTED_SHA,
        "global_alpha": SELECTED_ALPHA,
        "development_selection":
            selected_row.to_dict(),
        "monitoring":
            MONITOR_SUMMARY,
        "selection_used_monitoring": False,
    },
)

print(
    "\nGLOBAL_ALPHA is now fixed at",
    GLOBAL_ALPHA,
    "for the submission-generation cell.",
)


Evaluating: teacher_epoch01_alpha_065
GLOBAL_ALPHA: 0.65


Always-on enhancement:   0%|          | 0/104 [00:00<?, ?it/s]

Always-on enhancement:   0%|          | 0/100 [00:00<?, ?it/s]

Content-audited SraVaani:   0%|          | 0/104 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:1790: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1479.)
  return forward_call(*args, **kwargs)


{'candidate': 'teacher_epoch01_alpha_065', 'subset': 'synthetic_dev', 'clips': 104, 'skipped': 0, 'reference_words': 1321, 'raw_errors': 458, 'enhanced_errors': 434, 'raw_wer_percent': np.float64(34.67070401211203), 'enhanced_wer_percent': np.float64(32.85389856169569), 'identical_transcript_fraction': 0.4423076923076923, 'delta_wer_points': np.float64(1.8168054504163464), 'raw_substitutions': 217, 'raw_deletions': 234, 'raw_insertions': 7, 'enhanced_substitutions': 230, 'enhanced_deletions': 194, 'enhanced_insertions': 10, 'clean_wer_percent': np.float64(22.937168811506435), 'enhanced_si_sdr': 7.829984837034754, 'noisy_si_sdr': 0.41693136310790574}


... (65 outputs left out)

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:1790: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1479.)
  return forward_call(*args, **kwargs)


{'candidate': 'teacher_epoch01_alpha_095', 'subset': 'natural_monitor', 'clips': 100, 'skipped': 0, 'reference_words': 1468, 'raw_errors': 480, 'enhanced_errors': 474, 'raw_wer_percent': np.float64(32.69754768392371), 'enhanced_wer_percent': np.float64(32.28882833787466), 'identical_transcript_fraction': 0.81, 'delta_wer_points': np.float64(0.4087193460490468), 'raw_substitutions': 225, 'raw_deletions': 245, 'raw_insertions': 10, 'enhanced_substitutions': 221, 'enhanced_deletions': 243, 'enhanced_insertions': 10}


,selected_alpha,synthetic_si_sdr,synthetic_delta_wer_points,natural_delta_wer_points,pooled_delta_wer_points,combined_proxy,synthetic_deletion_change,natural_deletion_change
0,0.95,14.599056,-1.020408,0.408719,-0.255288,14.343768,1,-2



GLOBAL_ALPHA is now fixed at 0.95 for the submission-generation cell.


In [23]:
# Correct selection: maximize the actual combined competition proxy.
GLOBAL_ALPHA = 1.00

SELECTED = RUN / "teacher_epoch_01.pt"
if not SELECTED.is_file():
    raise FileNotFoundError(SELECTED)

SELECTED_SHA = sha_file(SELECTED)
BEST = "teacher_epoch01_alpha_100"

selected_row = SWEEP_RESULTS.loc[
    np.isclose(
        SWEEP_RESULTS["global_alpha"].astype(float),
        GLOBAL_ALPHA,
    )
].sort_values(
    "combined_proxy",
    ascending=False,
).iloc[0]

put_json(
    RUN / "teacher_epoch01_selected_alpha.json",
    {
        "candidate": BEST,
        "checkpoint": str(SELECTED),
        "checkpoint_sha256": SELECTED_SHA,
        "global_alpha": GLOBAL_ALPHA,
        "development_selection": selected_row.to_dict(),
        "selection_rule":
            "maximum development combined proxy; "
            "monitoring not used for selection",
    },
)

print("Corrected selected candidate:", BEST)
print("GLOBAL_ALPHA:", GLOBAL_ALPHA)
print(
    "Development combined proxy:",
    float(selected_row["combined_proxy"]),
)
print(
    "Expected monitor proxy from the previous evaluation:",
    16.48024536817611
    - (19 / (1274 + 1468) * 100),
)

Corrected selected candidate: teacher_epoch01_alpha_100
GLOBAL_ALPHA: 1.0
Development combined proxy: 18.463262351630352
Expected monitor proxy from the previous evaluation: 15.787320495820165


In [ ]:
# Generate the final Phase-2 submission using teacher_epoch_01 at alpha=1.00.

GLOBAL_ALPHA = 1.00
SIGN_ENSEMBLE = True

SELECTED = RUN / "teacher_epoch_01.pt"

if not SELECTED.is_file():
    raise FileNotFoundError(SELECTED)

SELECTED_SHA = sha_file(SELECTED)
BEST = "teacher_epoch01_alpha_100"

if not CONFIRM_RULES_ALLOW_SOURCE_DERIVED_TRAINING:
    raise RuntimeError(
        "Source-derived training permission confirmation is required."
    )

submission_ids = sorted(TRANSCRIPT_MAP)

if len(submission_ids) != EXPECTED_PHASE2_CLIPS:
    raise ValueError(
        f"Expected {EXPECTED_PHASE2_CLIPS} Phase-2 clips, "
        f"found {len(submission_ids)}"
    )


# Load all Phase-2 source audio.
source = StrictWaveSource(
    SOURCE_AUDIO,
    submission_ids,
)


# Free GPU memory used by SraVaani before enhancement.
asr_model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

activate(SELECTED)


submission_wavs = (
    WORK
    / "phase2_submission"
    / BEST
    / SELECTED_SHA[:12]
)

submission_wavs.mkdir(
    parents=True,
    exist_ok=True,
)


paths = {}
frames = {}
processing_audit = []


for clip_id in tqdm(
    submission_ids,
    desc="Final all-clip always-on enhancement",
):
    noisy = source.load(clip_id)

    enhanced = enhance_audio(noisy)

    if enhanced.shape != noisy.shape:
        raise RuntimeError(
            f"{clip_id}: output length changed "
            f"from {len(noisy)} to {len(enhanced)}"
        )

    if not np.isfinite(enhanced).all():
        raise RuntimeError(
            f"{clip_id}: output contains NaN or Inf"
        )

    output_path = submission_wavs / f"{clip_id}.wav"

    write_pcm(
        output_path,
        enhanced,
    )

    output_sha = wav_audit(
        output_path,
        len(noisy),
    )

    unchanged_pcm = bool(
        np.array_equal(
            canonical_pcm(noisy),
            canonical_pcm(enhanced),
        )
    )

    change_rms = float(
        np.sqrt(
            np.mean(
                (
                    pcm_roundtrip(enhanced)
                    - pcm_roundtrip(noisy)
                )
                ** 2
            )
        )
    )

    processing_audit.append(
        {
            "clip_id": clip_id,
            "input_sha256": hashlib.sha256(
                noisy.tobytes()
            ).hexdigest(),
            "wav_sha256": output_sha,
            "checkpoint_sha256": SELECTED_SHA,
            "global_alpha": GLOBAL_ALPHA,
            "sign_ensemble": SIGN_ENSEMBLE,
            "model_calls": LAST_MODEL_CALLS,
            "unchanged_pcm": unchanged_pcm,
            "change_rms": change_rms,
            "clipped_samples": int(
                np.count_nonzero(
                    (enhanced < -1.0)
                    | (enhanced > 32767 / 32768)
                )
            ),
        }
    )

    paths[clip_id] = output_path
    frames[clip_id] = len(noisy)


source.close()


audit_frame = pd.DataFrame(
    processing_audit
)

audit_path = (
    RUN
    / "phase2_submission_processing_audit.csv"
)

audit_frame.to_csv(
    audit_path,
    index=False,
)


unchanged_count = int(
    audit_frame["unchanged_pcm"].sum()
)

if unchanged_count:
    unchanged_examples = audit_frame.loc[
        audit_frame["unchanged_pcm"],
        "clip_id",
    ].head(20).tolist()

    raise RuntimeError(
        f"{unchanged_count} outputs are PCM-identical to their inputs. "
        f"Examples: {unchanged_examples}. "
        "No unchanged-output fallback is permitted."
    )


if (
    audit_frame["model_calls"].astype(int) <= 0
).any():
    raise RuntimeError(
        "At least one clip was not processed by the neural model."
    )


print(
    "Enhancement audit PASS | clips:",
    len(audit_frame),
)

print(
    "Mean change RMS:",
    float(audit_frame["change_rms"].mean()),
)

print(
    "Clipped samples:",
    int(audit_frame["clipped_samples"].sum()),
)


# Release enhancer and load SraVaani for submission transcripts.
model.to("cpu")
gc.collect()
torch.cuda.empty_cache()

asr_model.to(DEVICE).eval()

submission_transcripts = ASR.transcribe(
    paths
)


submission_zip = RUN / (
    "submission_Track2_PHASE2_V36_"
    "Phase1Teacher_"
    f"{BEST}_"
    f"{SELECTED_SHA[:10]}.zip"
)


package_report = pack_submission(
    paths,
    submission_transcripts,
    frames,
    submission_zip,
)


manifest = {
    "version":
        "v36_phase1_dictionary_teacher_demucs",
    "candidate":
        BEST,
    "checkpoint":
        str(SELECTED),
    "checkpoint_sha256":
        SELECTED_SHA,
    "source_checkpoint_sha256":
        SOURCE_SHA,
    "global_alpha":
        GLOBAL_ALPHA,
    "sign_ensemble":
        SIGN_ENSEMBLE,
    "all_clips_enhanced":
        True,
    "unchanged_outputs":
        unchanged_count,
    "routing":
        False,
    "classifier_routing":
        False,
    "dictionary_inference":
        False,
    "beats_in_inference":
        False,
    "tts_in_inference":
        False,
    "continuous_noise_conditioning":
        True,
    "residual_subtraction":
        True,
    "phase1_dictionary_teacher_training":
        True,
    "teacher_pairs":
        int(len(TEACHER)),
    "source_derived_training_permission_confirmed":
        True,
    "processing_audit":
        str(audit_path),
    "submission_zip":
        str(submission_zip),
    "package":
        package_report,
}


put_json(
    RUN / "submission_manifest.json",
    manifest,
)


print("\nSubmission ready:")
print(submission_zip)
print("\nManifest:")
print(RUN / "submission_manifest.json")
print("\nProcessing audit:")
print(audit_path)

Final all-clip always-on enhancement:   0%|          | 0/3178 [00:00<?, ?it/s]

Enhancement audit PASS | clips: 3178
Mean change RMS: 0.04147651535144654
Clipped samples: 5832


Content-audited SraVaani:   0%|          | 0/3178 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/module.py:1790: UserWarning: RNN module weights are not part of single contiguous chunk of memory. This means they need to be compacted at every call, possibly greatly increasing memory usage. To compact weights again call flatten_parameters(). (Triggered internally at /pytorch/aten/src/ATen/native/cudnn/RNN.cpp:1479.)
  return forward_call(*args, **kwargs)



Submission ready:
/content/drive/MyDrive/PHD/Datathon/track2_phase2_v36_phase1_teacher_demucs/run_v1/submission_Track2_PHASE2_V36_Phase1Teacher_teacher_epoch01_alpha_100_e95c5eb8ca.zip

Manifest:
/content/drive/MyDrive/PHD/Datathon/track2_phase2_v36_phase1_teacher_demucs/run_v1/submission_manifest.json

Processing audit:
/content/drive/MyDrive/PHD/Datathon/track2_phase2_v36_phase1_teacher_demucs/run_v1/phase2_submission_processing_audit.csv


## Outputs

All V36 artifacts are written under:

`/content/drive/MyDrive/PHD/Datathon/track2_phase2_v36_phase1_teacher_demucs/run_v1`

The original V35 epoch-3 checkpoint remains a candidate. The Phase-1
dictionary is used only to create offline training targets; it is absent from
the submitted inference pipeline. Unchanged teacher outputs are excluded and
reported by the extraction audit.
